In [1]:
import pandas as pd
import numpy as np

In [2]:
ramen_rating = pd.read_csv(r"C:\Users\sukai\Downloads\archive (5).zip")
ramen_rating.head()

,Review #,Brand,Variety,Style,Country,Stars,Top Ten
0,2580,New Touch,T's Restaurant Tantanmen,Cup,Japan,3.75,NaN
1,2579,Just Way,Noodles Spicy Hot Sesame Spicy Hot Sesame Guan...,Pack,Taiwan,1,NaN
2,2578,Nissin,Cup Noodles Chicken Vegetable,Cup,USA,2.25,NaN
3,2577,Wei Lih,GGE Ramen Snack Tomato Flavor,Pack,Taiwan,2.75,NaN
4,2576,Ching's Secret,Singapore Curry,Pack,India,3.75,NaN


In [10]:
from sqlalchemy import create_engine
engine = create_engine("sqlite://")
ramen_rating.to_sql("my_table", engine, index = False)

2580

In [3]:
#create a "Series" with index as stars and whose values count how many stars they got
stars_given = ramen_rating.groupby("Stars")["Stars"].count()
stars_given.head()

Stars
0       26
0.1      1
0.25    11
0.5     14
0.75     1
Name: Stars, dtype: int64

In [4]:
#create a "Series" with index as review and whose values is the maximum no. of stars given by the reviewers.sort the values by stars in ascending order
review_given = ramen_rating.groupby("Review #")["Stars"].max().sort_values()
review_given

Review #
2527          0
498           0
1628          0
2017          0
2056          0
         ...   
2126       5.00
2133       5.00
2548    Unrated
1587    Unrated
2458    Unrated
Name: Stars, Length: 2580, dtype: object

In [5]:
#create a "DataFrame" with index as variety whose values are the min and max values
variety_df = ramen_rating.groupby("Variety")["Stars"].agg(["min", "max"])
variety_df

,min,max
Variety,,
"""A"" Series Artificial Chicken",3.25,3.25
"""A"" Series Artificial Hot Beef",3.75,3.75
"""A"" Series Vegetarian",3.25,3.25
1 Step-1 Minute Asian Noodles Kung Pao,4.25,4.25
1 Step-1 Minute Asian Noodles Lemongrass Ginger,3.75,3.75
...,...,...
chicken,3.25,3.25
dried Mix Noodles Soya Bean Paste,3.75,3.75
spicy Pad Thai Instant Noodles & Sauce,4.25,4.25


In [6]:
#create a "DataFrame" using the above variety_df with the stars in descending order sorted by minimum then maximum
variety_df_desc = variety_df.sort_values(by = ["min", "max"], ascending = False)
variety_df_desc

,min,max
Variety,,
Plain Instant Noodle No Soup Included,Unrated,Unrated
Plain Noodles,Unrated,Unrated
Sari Ramen,Unrated,Unrated
Cup Noodle Big Cheese Mexican Chilli,5.00,5.00
Hong Kong Street Beef,5.00,5.00
...,...,...
Spicy Tomato Salsa Ramen,0,0
Sweet Potato Instant Noodle Sout-Hot Flavor,0,0
Tiny Noodle With Oyster Flavor,0,0


In [7]:
#create a "Series" whose index is the country and whose values is the average Review # given
review_given = ramen_rating.groupby("Country")["Review #"].mean()
review_given.head()

Country
Australia     2005.000000
Bangladesh    1953.857143
Brazil        2093.600000
Cambodia      1822.400000
Canada        1361.512195
Name: Review #, dtype: float64

In [8]:
#create a "Series" whose multi-index is the country and variety and sort the values in descending order by the ramen count
ramen_rating_multi_index = ramen_rating.groupby(["Country", "Variety"]).size().sort_values(ascending = False)
ramen_rating_multi_index
#in this code we use "size()" method so that it doesn't show error for not using the "by" arguement inside sort_values() method.
#if we use "count()" method instead of "size()" method, the code will count the data one by one which will lead to a a mess of data while sorting.

Country      Variety                                  
Japan        Yakisoba                                     4
             Curry Udon                                   4
China        Artificial Spicy Beef                        3
South Korea  Beef                                         3
             Kokomen Spicy Chicken                        3
                                                         ..
Australia    Noodles With Chicken & Sweet Corn Flavour    1
             Noodles With Chicken Flavour                 1
             Noodles With Hot & Spicy Flavour             1
             Noodles Witrh Prawn & Chicken Flavour        1
             Chow Mein Soft Noodles                       1
Length: 2486, dtype: int64

In [16]:
query = """
SELECT country, SUM(stars) FROM my_table
GROUP BY country
HAVING country = "Taiwan";
"""
sql_my_table = pd.read_sql_query(query, con=engine)
print(sql_my_table.head())

  Country  SUM(stars)
0  Taiwan      821.05
